In [3]:
import sys
!{sys.executable} -m pip install yfinance

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.6/84.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 144.3 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.8/116.8 kB 275.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 201.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 305.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.2/112.2 kB 245.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 175.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 227.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.4/345.4 kB 312.0 MB/s eta 0:00:00
  Created wheel for peewee: filename=peewee-3.17.6-cp39-cp39-linux_x86_64.whl size=753132 sha256=72568e0d3677ca6e1b4a047

In [21]:
import yfinance as yfin
import pandas as pd
import csv
import numpy as np

In [6]:
df = yfin.download(tickers=['AAPL'], period='6mo')
#df = yfin.download(tickers=ticker, period='6mo')
dataset = df['Close'].fillna(method='ffill')
dataset = dataset.values.reshape(-1, 1)

[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_874/4288447120.py:3: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dataset = df['Close'].fillna(method='ffill')


In [7]:
dataset.shape

(127, 1)

In [14]:
data = pd.DataFrame(dataset)
data.to_csv('file1.csv', header=False, index=False)

In [22]:
with open('file1.csv', 'r') as f:
    reader = csv.reader(f)
    dataset1 = list(reader)
data_array = np.array(dataset1)

In [24]:
data_array.shape

(127, 1)

In [ ]:

scaler = MinMaxScaler(feature_range=(0, 1))
scaler = scaler.fit(dataset1)
dataset = scaler.transform(dataset1)

In [ ]:
# generate the input and output sequences
n_lookback = 60  # length of input sequences (lookback period)
n_forecast = 30  # length of output sequences (forecast period)

X = []
Y = []

for i in range(n_lookback, len(dataset) - n_forecast + 1):
    X.append(dataset[i - n_lookback: i])
    Y.append(dataset[i: i + n_forecast])

In [ ]:
X = np.array(X)
Y = np.array(Y)


In [ ]:
# fit the model
model = Sequential(name="forecast")
model.add(LSTM(units=50, return_sequences=True, input_shape=(n_lookback, 1)))
model.add(LSTM(units=50))
model.add(Dense(n_forecast))

In [ ]:

model.compile(loss='mean_squared_error', optimizer='adam')
model.fit(X, Y, epochs=100, batch_size=32, verbose=0)

In [ ]:
# generate the forecasts
X_ = dataset[- n_lookback:]  # last available input sequence
X_ = X_.reshape(1, n_lookback, 1)

Y_ = model.predict(X_).reshape(-1, 1)
Y_ = scaler.inverse_transform(Y_)

In [ ]:
# organize the results in a data frame
#df_past = df[['Close']].reset_index()
#df_past.rename(columns={'index': 'Date', 'Close': 'Actual'}, inplace=True)
#df_past['Date'] = pd.to_datetime(df_past['Date'])
#df_past['Forecast'] = np.nan
#df_past['Forecast'].iloc[-1] = df_past['Actual'].iloc[-1]

In [ ]:

#df_future = pd.DataFrame(columns=['Date', 'Actual', 'Forecast'])
#df_future['Date'] = pd.date_range(start=df_past['Date'].iloc[-1] + pd.Timedelta(days=1), periods=n_forecast)
#df_future['Forecast'] = Y_.flatten()
#df_future['Actual'] = np.nan

In [ ]:
#results = pd.concat([df_past, df_future])
#results = results.set_index('Date')

# plot the results
#results.plot(title='IBM')


In [ ]:
import os
model.save("./forecast.keras")

In [ ]:
onnx_model, _ = tf2onnx.convert.from_keras(model)
onnx.save(onnx_model, "./forecast.onnx")

In [ ]:
client = Minio(
    "minio.stock-predict.svc.cluster.local:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)


In [ ]:
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

In [ ]:
bucket_name = "models"
source_file = "./forecast.onnx"
destination_file = "forecast.onnx"
client.fput_object(bucket_name, destination_file, source_file)